# Norm

normalization，即标准化 / 归一化。它的核心作用是：

> 调整神经网络中数据的尺度和分布，让训练更加稳定、更容易优化。

在神经网络训练过程中，我们经常会遇到几个问题。

1. 梯度不稳定

神经网络的参数会根据梯度反向传播进行更新。更标准的参数更新形式是：

$$
W_{t+1} = W_t - \eta \frac{\partial L}{\partial W}
$$

其中 $\eta$ 是 learning rate，$L$ 是 loss。如果中间激活值的尺度非常大或非常小，那么反向传播时梯度也可能变得过大或过小，从而导致梯度爆炸或梯度消失。

2. 激活函数饱和

以 Sigmoid 激活函数为例，当输入值非常大或者非常小时，Sigmoid 的输出会接近 1 或 0，此时梯度非常小，参数就很难继续被有效更新。normalization 可以把输入值压到一个更稳定的范围内，减少进入饱和区间的可能。

3. 输入分布偏移

在训练过程中，前一层参数不断变化，后一层看到的输入分布也会不断变化。这样会让优化过程更困难。normalization 的作用之一，就是让每一层接收到的数据分布更加稳定。

## normalization 的基本公式

normalization 通常先对输入减去均值，再除以标准差：

$$
\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}
$$

其中：

$$
\mu = \mathbb{E}[x], \quad \sigma^2 = \operatorname{Var}(x)
$$

$\epsilon$ 是一个很小的正数，主要用于防止分母为 0，保证数值稳定。

经过这一步后，$\hat{x}$ 大致会变成均值为 0、方差为 1 的分布：

$$
\mathbb{E}[\hat{x}] \approx 0, \quad \operatorname{Var}(\hat{x}) \approx 1
$$

但如果只做这一步，模型的表达能力可能会受限。因为有时候模型确实需要某些特征保持更大的尺度，或者有一个非 0 的偏移。因此 normalization 后面通常还会接一个可学习的缩放和平移：

$$
y = \gamma \hat{x} + \beta
$$

其中 $\gamma$ 和 $\beta$ 是可学习参数。它们的作用可以理解为：先把数据标准化到稳定范围内，再让模型自己学习是否需要把它缩放或平移回某种更合适的分布。

所以完整写法是：

$$
y = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta
$$

## BatchNorm 和 LayerNorm

假设输入为：

$$
x \in \mathbb{R}^{B \times L \times C}
$$

其中 $B$ 是 batch size，$L$ 是 sequence length，$C$ 是 hidden size / channel size。

BatchNorm 和 LayerNorm 的核心区别是：它们统计均值和方差的维度不同。

## LayerNorm

LayerNorm 对每个 token 单独做归一化。也就是说，对于每一个 $(B,L)$ 位置，它会在 $C$ 个 feature 上统计均值和方差：

$$
\mu_{b,l} = \frac{1}{C}\sum_{c=1}^{C} x_{b,l,c}
$$

$$
\sigma^2_{b,l} = \frac{1}{C}\sum_{c=1}^{C}(x_{b,l,c} - \mu_{b,l})^2
$$

因此 LayerNorm 的 mean/std shape 是：

$$
[B, L, 1]
$$

它的可学习参数 $\gamma$ 和 $\beta$ 作用在 channel 维度上：

$$
\gamma, \beta \in \mathbb{R}^{C}
$$

也就是说，不同 token 的 mean/std 可以不同，但同一个 channel 的缩放和平移参数是共享的。LayerNorm 的统计过程与 batch size 无关，因此在 NLP、Transformer、LLM 中非常常见。

## BatchNorm

BatchNorm 对每个 channel 单独做归一化。对于输入 $x \in \mathbb{R}^{B \times L \times C}$，它会在 $B \times L$ 个位置上统计同一个 channel 的均值和方差：

$$
\mu_c = \frac{1}{B L}\sum_{b=1}^{B}\sum_{l=1}^{L}x_{b,l,c}
$$

$$
\sigma^2_c = \frac{1}{B L}\sum_{b=1}^{B}\sum_{l=1}^{L}(x_{b,l,c} - \mu_c)^2
$$

因此 BatchNorm 的 mean/std shape 是：

$$
[1, 1, C]
$$

它的可学习参数同样是：

$$
\gamma, \beta \in \mathbb{R}^{C}
$$

也就是说，所有 batch、所有 token 在同一个 channel 上共享同一个 mean/std。BatchNorm 在 CNN 中很常见，因为图像任务里 batch 统计通常比较稳定。但在序列模型和大语言模型中，batch size、sequence length 变化较多，所以 LayerNorm 往往更常用。

需要注意：实际 PyTorch 的 `BatchNorm` 在训练时会使用 batch 统计量，在推理时会使用 running mean / running variance。本 notebook 下面实现的是一个简化版本，只展示训练时如何根据当前 batch 计算 mean 和 variance。

参考图片：

![LayerNorm 和 BatchNorm 对比](../figs/norm.png)


In [7]:
# LayerNorm and BatchNorm implementation
import torch
import torch.nn as nn

# weight and bias 只与 hidden_size 相关，和 batch_size、seq_len 无关

class LayerNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))  # gamma, shape: [hidden_size]
        self.bias = nn.Parameter(torch.zeros(hidden_size))   # beta, shape: [hidden_size]
        self.eps = eps

    def forward(self, x):
        # x: [batch_size, seq_len, hidden_size]
        mean = x.mean(dim=-1, keepdim=True)  # [batch_size, seq_len, 1]
        var = x.var(dim=-1, keepdim=True, unbiased=False)  # [batch_size, seq_len, 1]

        # 对每个 token，在 hidden_size 维度上计算自己的 mean/var
        x_norm = (x - mean) / torch.sqrt(var + self.eps)  # [batch_size, seq_len, hidden_size]

        return self.weight * x_norm + self.bias

class BatchNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))  # gamma, shape: [hidden_size]
        self.bias = nn.Parameter(torch.zeros(hidden_size))   # beta, shape: [hidden_size]
        self.eps = eps

    def forward(self, x):
        # x: [batch_size, seq_len, hidden_size]
        mean = x.mean(dim=(0, 1), keepdim=True)  # [1, 1, hidden_size]
        var = x.var(dim=(0, 1), keepdim=True, unbiased=False)  # [1, 1, hidden_size]

        # 对每个 channel，在 batch_size 和 seq_len 维度上共享 mean/var
        x_norm = (x - mean) / torch.sqrt(var + self.eps)  # [batch_size, seq_len, hidden_size]

        return self.weight * x_norm + self.bias


In [8]:
# test the implementation
torch.manual_seed(0)

x = torch.randn(2, 3, 4)  # [batch_size, seq_len, hidden_size]
ln = LayerNorm(hidden_size=4)
bn = BatchNorm(hidden_size=4)

ln_out = ln(x)
bn_out = bn(x)

print("input shape:", x.shape)
print("LayerNorm output shape:", ln_out.shape)
print("BatchNorm output shape:", bn_out.shape)

# LayerNorm: 每个 token 在 hidden_size 维度上的均值接近 0，方差接近 1
print("LayerNorm mean over hidden dim:")
print(ln_out.mean(dim=-1))
print("LayerNorm var over hidden dim:")
print(ln_out.var(dim=-1, unbiased=False))

# BatchNorm: 每个 channel 在 batch_size 和 seq_len 维度上的均值接近 0，方差接近 1
print("BatchNorm mean over batch/seq dims:")
print(bn_out.mean(dim=(0, 1)))
print("BatchNorm var over batch/seq dims:")
print(bn_out.var(dim=(0, 1), unbiased=False))


input shape: torch.Size([2, 3, 4])
LayerNorm output shape: torch.Size([2, 3, 4])
BatchNorm output shape: torch.Size([2, 3, 4])
LayerNorm mean over hidden dim:
tensor([[1.4901e-07, 2.9802e-08, 4.4703e-08],
        [1.4901e-08, 5.9605e-08, 0.0000e+00]], grad_fn=<MeanBackward1>)
LayerNorm var over hidden dim:
tensor([[0.9999, 1.0000, 1.0000],
        [0.9999, 1.0000, 1.0000]], grad_fn=<VarBackward0>)
BatchNorm mean over batch/seq dims:
tensor([-2.9802e-08,  2.9802e-08,  0.0000e+00,  0.0000e+00],
       grad_fn=<MeanBackward1>)
BatchNorm var over batch/seq dims:
tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<VarBackward0>)
